# Demonstrasi: Fitur, Pembagian Data, dan Baseline — PIF304 Pengantar Data Sains
**Pertemuan 10 — Sub-CPMK S04: Menerapkan fitur dan baseline analitik**

Sebelum model apa pun dilatih, empat hal harus beres:

1. **Tugas prediksi** dirumuskan dengan jelas (target, unit, *kapan* prediksi dibuat)
2. **Fitur** dipilih dengan alasan — dan yang bocor (*leakage*) disingkirkan
3. **Data dibagi** menjadi data latih dan data uji **sebelum** ada penghitungan apa pun
4. **Baseline** ditetapkan sebagai pembanding minimum

**Dataset:** `Data_Akademik_Sintetis_PDS.xlsx` — **data sintetis**, 600 baris.

**Tugas prediksi:** pada **akhir semester 2**, memperkirakan mahasiswa mana yang akan berstatus *"Perlu pendampingan"* di akhir semester 4, agar pendampingan dapat ditawarkan lebih awal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_absolute_error, confusion_matrix
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Akademik_Sintetis_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Akademik_Sintetis_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Akademik_Sintetis_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

## 1. Rumuskan Tugas Prediksi
| Unsur | Rumusan |
|---|---|
| Unit analisis | 1 mahasiswa |
| Target (klasifikasi) | `status_sem4` → 1 = Perlu pendampingan, 0 = Aman |
| Target (regresi, pembanding) | `ipk_sem4` |
| Waktu prediksi | akhir semester 2 |
| Konsekuensi | hanya informasi yang **sudah tersedia di akhir semester 2** boleh menjadi fitur |

In [ ]:
y_kelas = (df["status_sem4"] == "Perlu pendampingan").astype(int)
y_reg = df["ipk_sem4"]
print("Proporsi kelas positif (Perlu pendampingan):", round(y_kelas.mean(), 3))

## 2. Memilih Fitur: Empat Pertanyaan Saringan
Untuk setiap kolom, tanyakan:
1. **Tersedia saat prediksi dibuat?** (bukan direkam setelahnya)
2. **Relevan?** (ada dasar dari EDA/pengetahuan domain)
3. **Mutunya memadai?** (kosong sedikit, definisinya jelas)
4. **Etis dipakai?** (bukan atribut sensitif yang dapat menimbulkan perlakuan tidak adil)

In [ ]:
saringan = pd.DataFrame([
    # kolom,                 tersedia, relevan, mutu,  etis,  keputusan, alasan
    ["id_mhs",                 "ya",  "tidak", "ya",  "ya",   "buang", "Pengenal, bukan informasi"],
    ["ips_sem1",               "ya",  "ya",    "ya",  "ya",   "pakai", "Korelasi kuat dengan target (EDA)"],
    ["ips_sem2",               "ya",  "ya",    "ya",  "ya",   "pakai", "Korelasi terkuat (EDA)"],
    ["sks_lulus_tahun1",       "ya",  "ya",    "ya",  "ya",   "pakai", "Kemajuan studi tahun pertama"],
    ["kehadiran_pct",          "ya",  "ya",    "cukup", "ya", "pakai", "2 nilai > 100 dijadikan kosong"],
    ["jam_belajar_per_minggu", "ya",  "ya",    "cukup", "ya", "pakai", "Lapor diri; 4% kosong"],
    ["akses_lms_per_minggu",   "ya",  "ya",    "ya",  "ya",   "pakai", "Jejak aktivitas belajar"],
    ["tugas_terlambat",        "ya",  "ya",    "ya",  "ya",   "pakai", "Indikator kedisiplinan"],
    ["bekerja_paruh_waktu",    "ya",  "mungkin", "ya", "hati-hati", "pakai", "Dapat menjadi dasar dukungan, bukan hukuman"],
    ["prodi",                  "ya",  "tidak", "ya",  "ya",   "buang", "Rata-rata IPK antarprodi hampir sama (EDA)"],
    ["angkatan",               "ya",  "tidak", "ya",  "ya",   "buang", "Tidak ada dasar relevansi"],
    ["jalur_masuk",            "ya",  "mungkin", "ya", "hati-hati", "buang", "Ditunda; informasinya sudah tercermin pada IPS"],
    ["asal_daerah",            "ya",  "tidak", "ya",  "hati-hati", "buang", "Tidak ada dasar relevansi"],
    ["jarak_km",               "ya",  "tidak", "cukup", "ya", "buang", "Korelasi mendekati nol (EDA)"],
    ["usia_masuk",             "ya",  "tidak", "ya",  "ya",   "buang", "Korelasi mendekati nol (EDA)"],
    ["jenis_kelamin",          "ya",  "tidak", "ya",  "TIDAK", "buang", "Atribut sensitif; tidak ada dasar relevansi"],
    ["penerima_beasiswa",      "ya",  "semu",  "ya",  "hati-hati", "buang", "Efeknya berasal dari seleksi IPS (EDA P9)"],
    ["surat_peringatan_sem4",  "TIDAK", "-",   "-",   "-",    "buang", "Direkam di akhir sem 4 = BOCOR"],
    ["ipk_sem4",               "TIDAK", "-",   "-",   "-",    "buang", "Ini target (regresi) / penentu target (klasifikasi)"],
], columns=["kolom", "tersedia_saat_prediksi", "relevan", "mutu", "etis", "keputusan", "alasan"])
saringan

## 3. Data Leakage: "Terlalu Bagus untuk Benar"
*Leakage* terjadi bila model mendapat informasi yang **tidak akan tersedia** saat prediksi sungguhan dibuat. Tanda bahayanya: hasil yang nyaris sempurna.

In [ ]:
print(pd.crosstab(df["surat_peringatan_sem4"], df["status_sem4"]), "\n")

tebak_dari_sp = (df["surat_peringatan_sem4"] == "Ya").astype(int)
print("'Model' satu baris: jika dapat surat peringatan -> Perlu pendampingan")
print(f"  accuracy  = {accuracy_score(y_kelas, tebak_dari_sp):.3f}")
print(f"  precision = {precision_score(y_kelas, tebak_dari_sp):.3f}")
print(f"  recall    = {recall_score(y_kelas, tebak_dari_sp):.3f}")

Akurasi 91% dan precision 100% **tanpa model sama sekali**. Tetapi aturan ini tidak berguna: surat peringatan baru terbit di akhir semester 4, pada saat yang sama dengan target. Di akhir semester 2 kolom ini **belum ada**.

Dua bentuk kebocoran yang paling sering terjadi:

| Bentuk | Contoh | Pencegahan |
|---|---|---|
| **Kebocoran target** | Fitur direkam setelah/bersamaan dengan target (`surat_peringatan_sem4`) | Periksa *waktu perekaman* setiap kolom di data dictionary |
| **Kontaminasi latih–uji** | Imputasi/scaling dihitung dari seluruh data sebelum dibagi | **Bagi dulu**, baru hitung parameter dari data latih saja |

## 4. Menyiapkan Fitur

In [ ]:
fitur_numerik = ["ips_sem1", "ips_sem2", "sks_lulus_tahun1", "kehadiran_pct", "jam_belajar_per_minggu", "akses_lms_per_minggu", "tugas_terlambat"]

X = df[fitur_numerik].copy()
# (a) nilai mustahil -> kosong (aturan dari data dictionary; tidak bergantung pada data lain, jadi aman dilakukan sebelum pembagian)
X.loc[X["kehadiran_pct"] > 100, "kehadiran_pct"] = np.nan
X.loc[X["jam_belajar_per_minggu"] > 40, "jam_belajar_per_minggu"] = np.nan
# (b) encoding biner
X["bekerja"] = (df["bekerja_paruh_waktu"] == "Ya").astype(int)
# (c) fitur rekayasa sederhana
X["tren_ips"] = X["ips_sem2"] - X["ips_sem1"]            # naik (+) atau turun (-) dari sem 1 ke sem 2
X["kehadiran_rendah"] = (X["kehadiran_pct"] < 75).astype(int)

print("Jumlah fitur:", X.shape[1])
X.describe().T[["count", "mean", "min", "max"]].round(2)

## 5. Membagi Data: Latih vs Uji
Data uji adalah **simulasi masa depan**: data yang belum pernah dilihat model. Karena itu ia disisihkan **sekarang** dan baru dibuka saat evaluasi.

- `test_size=0.2` → 80% latih, 20% uji
- `random_state=42` → pembagian dapat diulang persis (*reproducibility*)
- `stratify=y_kelas` → proporsi kelas di kedua bagian tetap sama

In [ ]:
X_latih, X_uji, y_latih, y_uji, yreg_latih, yreg_uji = train_test_split(
    X, y_kelas, y_reg, test_size=0.2, random_state=42, stratify=y_kelas)

print("Latih:", X_latih.shape, "| proporsi positif:", round(y_latih.mean(), 3))
print("Uji  :", X_uji.shape, "| proporsi positif:", round(y_uji.mean(), 3))

### Imputasi setelah pembagian
Median untuk mengisi nilai kosong dihitung **dari data latih saja**, lalu dipakai pada keduanya.

In [ ]:
median_latih = X_latih.median()
X_latih = X_latih.fillna(median_latih)
X_uji = X_uji.fillna(median_latih)            # memakai median DATA LATIH, bukan median data uji
print("Nilai kosong tersisa -> latih:", int(X_latih.isna().sum().sum()), "| uji:", int(X_uji.isna().sum().sum()))
print("Median yang dipakai:", median_latih[["kehadiran_pct", "jam_belajar_per_minggu", "akses_lms_per_minggu"]].to_dict())

## 6. Baseline: Pembanding Minimum
Baseline menjawab: *"seberapa baik hasilnya jika kita hampir tidak berusaha?"* Model yang tidak mengungguli baseline tidak layak dipakai.

### 6a. Baseline klasifikasi

In [ ]:
def nilai_klasifikasi(nama, y_benar, y_tebak):
    return {"metode": nama, "accuracy": round(accuracy_score(y_benar, y_tebak), 3),
            "precision": round(precision_score(y_benar, y_tebak, zero_division=0), 3),
            "recall": round(recall_score(y_benar, y_tebak), 3)}

hasil = []
# Baseline 1: selalu menebak kelas terbanyak ("Aman")
mayoritas = DummyClassifier(strategy="most_frequent").fit(X_latih, y_latih)
hasil.append(nilai_klasifikasi("Baseline 1: selalu 'Aman'", y_uji, mayoritas.predict(X_uji)))

# Baseline 2: aturan sederhana dari EDA -> ips_sem2 < 2,75 berarti Perlu pendampingan
aturan = (X_uji["ips_sem2"] < 2.75).astype(int)
hasil.append(nilai_klasifikasi("Baseline 2: aturan ips_sem2 < 2,75", y_uji, aturan))

baseline_kelas = pd.DataFrame(hasil)
baseline_kelas

In [ ]:
print("Confusion matrix Baseline 2 (baris = sebenarnya, kolom = tebakan):")
pd.DataFrame(confusion_matrix(y_uji, aturan), index=["sebenarnya: Aman", "sebenarnya: Perlu"], columns=["tebak: Aman", "tebak: Perlu"])

**Membaca hasilnya:**
- Baseline 1 mencapai akurasi 71% **tanpa menemukan satu pun** mahasiswa yang perlu pendampingan (recall = 0). Akurasi tinggi pada kelas tidak seimbang bisa menipu.
- Baseline 2 (satu aturan dari EDA) sudah jauh lebih berguna. Inilah angka yang harus dikalahkan model di Pertemuan 11.

### 6b. Baseline regresi

In [ ]:
hasil_reg = []
rata = DummyRegressor(strategy="mean").fit(X_latih, yreg_latih)
hasil_reg.append({"metode": "Baseline 1: selalu rata-rata data latih", "MAE": round(mean_absolute_error(yreg_uji, rata.predict(X_uji)), 3)})
hasil_reg.append({"metode": "Baseline 2: ipk_sem4 = ips_sem2", "MAE": round(mean_absolute_error(yreg_uji, X_uji["ips_sem2"]), 3)})
baseline_reg = pd.DataFrame(hasil_reg)
baseline_reg

MAE (*mean absolute error*) = rata-rata besar selisih antara tebakan dan nilai sebenarnya, dalam satuan target. MAE 0,33 berarti tebakan "selalu rata-rata" meleset rata-rata 0,33 poin IPK.

## 7. Log Implementasi
Catatan singkat yang membuat pekerjaan dapat diulang dan diaudit — wajib pada mini-project (A6).

In [ ]:
log = pd.DataFrame([
    ["Tugas prediksi", "Klasifikasi status_sem4 pada akhir sem 2; positif = Perlu pendampingan (29%)"],
    ["Fitur", f"{X.shape[1]} fitur: " + ", ".join(X.columns)],
    ["Fitur dibuang", "id_mhs, prodi, angkatan, jalur_masuk, asal_daerah, jarak_km, usia_masuk, jenis_kelamin, penerima_beasiswa, surat_peringatan_sem4 (bocor), ipk_sem4"],
    ["Pembagian data", "80/20, stratify=y, random_state=42 -> 480 latih, 120 uji"],
    ["Penanganan kosong", "Nilai mustahil -> NaN; imputasi median data latih"],
    ["Baseline klasifikasi", f"mayoritas acc={baseline_kelas.loc[0,'accuracy']}; aturan ips_sem2<2,75 acc={baseline_kelas.loc[1,'accuracy']}, precision={baseline_kelas.loc[1,'precision']}, recall={baseline_kelas.loc[1,'recall']}"],
    ["Baseline regresi", f"rata-rata MAE={baseline_reg.loc[0,'MAE']}; ips_sem2 MAE={baseline_reg.loc[1,'MAE']}"],
], columns=["langkah", "keputusan"])
log.to_csv("log_implementasi_demo_P10.csv", index=False)
log

## Ringkasan
1. **Tulis tugas prediksinya** — termasuk *kapan* prediksi dibuat.
2. **Saring fitur** dengan empat pertanyaan: tersedia, relevan, bermutu, etis.
3. **Curigai hasil yang terlalu bagus** — periksa waktu perekaman.
4. **Bagi dulu, hitung kemudian** — parameter apa pun berasal dari data latih.
5. **Baseline dulu, model kemudian.**

**Berikutnya (Pertemuan 11):** melatih model regresi dan klasifikasi pertama, lalu membandingkannya dengan baseline hari ini.